# Meet the data: the emergency department tracking board

**Master in Healthcare Analytics · Dataset 2 of 5 · Introduction**

This notebook introduces the second dataset of the programme: one year of visits to the emergency department (ED) of Riverbend Community Hospital, as recorded by the ED **tracking board**. It follows the same plan as the outpatient introduction: the clinical and operational context, how to load the data, a first look at each table, and a column-by-column explanation of what every field means and where it would come from in a real hospital.

The big change from the outpatient dataset is **time**. Every visit here carries five timestamps to the minute, and most of the questions the data can answer are about waiting, crowding and throughput, and how they change over the hours of a day, the days of a week and the months of a year.

## 1. Context: where this data comes from

### The department

Riverbend Community Hospital is the same fictional 300-bed community hospital as in dataset 1. Its emergency department is open 24 hours a day and sees about **125 patients a day**, roughly 45,000 a year, which is typical of a mid-sized US community hospital. The ED has four treatment areas: the **Main** area, a **Resuscitation** bay for the sickest patients, a **Behavioral** area for psychiatric presentations, and, from 1 July 2025, a **Fast track** for minor problems, staffed mainly by physician assistants and nurse practitioners.

### The tracking board

Every US emergency department runs a tracking board: a live screen (and the database behind it) that shows every patient currently in the department, where they are, how long they have been there, and what they are waiting for. Each visit passes through five recorded events:

| Event | What happens | Timestamp column |
|---|---|---|
| 1. Arrival | The patient is registered at the front desk or handed over by ambulance crew | `arrival_datetime` |
| 2. Triage | A nurse assesses urgency, assigns an **ESI level** and records the chief complaint | `triage_datetime` |
| 3. Provider contact | First contact with a physician, resident or PA/NP | `provider_datetime` |
| 4. Disposition decision | The provider decides what happens next: discharge, admit, observe, transfer... | `decision_datetime` |
| 5. Departure | The patient physically leaves the ED | `departure_datetime` |

The gaps between these events are the department's core performance measures. **Door-to-provider** (arrival to provider contact) is the standard measure of responsiveness. **Boarding** (decision to departure, for admitted patients) is the time an admitted patient spends waiting in the ED for an inpatient bed, and it depends on the rest of the hospital rather than on the ED. **ED length of stay** (arrival to departure) is the total.

### Two things that happened in 2025

- On **1 July 2025** the ED opened its fast-track area for low-acuity patients (ESI 4 and 5), open 10:00–22:00. Whether it changed anything, and for whom, is a question the data can answer.
- In the **second week of February** a winter storm coincided with the influenza peak and produced the busiest week of the year.

### Why this matters clinically and operationally

ED crowding is one of the best-documented safety problems in hospital medicine: long waits are associated with delayed treatment of time-critical conditions, patients leaving without being seen, and worse outcomes for admitted patients who board for hours in ED corridors. Crowding is also mostly a *flow* problem rather than a *volume* problem: arrivals are highly predictable by hour and day, so the question is usually whether staffing and inpatient bed availability follow the arrival curve. That is why this dataset is built around time.

### What the data is, and is not

The data is **fully synthetic**: no row describes a real patient or visit. It was generated from a statistical model so that its patterns are realistic in size and direction. It is clean by design: no duplicates, no malformed timestamps, no impossible orderings. It does contain **missing values**, but only where an event never happened: a patient who left before being seen has no provider timestamp, because there was no provider contact. We will look at those in section 4.

## 2. Loading the data

Three CSV files, published in the course repository on GitHub. `pd.read_csv` can read a file directly from a URL, so nothing needs to be downloaded by hand. The `parse_dates` argument matters more than it did for the outpatient data: five columns hold date-and-time values to the minute, and pandas needs to know that so we can later subtract them, group by hour, and resample by day.

In [1]:
import pandas as pd

BASE_URL = "https://raw.githubusercontent.com/thousandoaks/Python4DS-II/main/datasets/EDDataset/"

timestamp_cols = ["arrival_datetime", "triage_datetime", "provider_datetime", "decision_datetime", "departure_datetime"]
visits = pd.read_csv(BASE_URL + "ed_visits.csv", parse_dates=timestamp_cols)
daily = pd.read_csv(BASE_URL + "ed_daily.csv", parse_dates=["date"])
patients = pd.read_csv(BASE_URL + "ed_patients.csv")

print("ed_visits:  ", visits.shape[0], "rows x", visits.shape[1], "columns")
print("ed_daily:   ", daily.shape[0], "rows x", daily.shape[1], "columns")
print("ed_patients:", patients.shape[0], "rows x", patients.shape[1], "columns")

ed_visits:   44934 rows x 31 columns
ed_daily:    365 rows x 16 columns
ed_patients: 18176 rows x 7 columns


| File | One row is... | Rows | Role |
|---|---|---|---|
| `ed_visits.csv` | one ED visit | 44,934 | The main table. Patient attributes are already joined. |
| `ed_daily.csv` | one calendar day | 365 | A ready-made daily time series, every column an aggregate of the visits table. |
| `ed_patients.csv` | one patient seen in the ED | 18,176 | Reference table; links to the outpatient `patients.csv` through `patient_id`. |

About 39% of the ED's patients (46% of visits) are people who also appear in the outpatient dataset, with the **same patient ID**, so the two datasets can be joined later. The others are patients the clinics have never seen.

## 3. A first look at each table

### 3.1 `ed_visits.csv`

The first ten rows are the first ten arrivals of the year, just after midnight on 1 January. Scroll horizontally: there are 31 columns, ordered roughly in the sequence events happen.

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
visits.head(10)

,visit_id,patient_id,arrival_datetime,arrival_mode,age,sex,primary_payer,triage_datetime,esi_level,chief_complaint,arrival_vitals_abnormal,ed_census_at_arrival,provider_datetime,provider_type,treatment_area,n_labs,n_imaging,ct_performed,decision_datetime,disposition,admit_unit_type,inpatient_occupancy_pct,departure_datetime,door_to_triage_min,door_to_provider_min,decision_to_departure_min,ed_los_min,arrival_date,arrival_hour,arrival_weekday,arrival_month
0,E000001,P115082,2025-01-01 00:04:00,Private vehicle,77,F,Medicare,2025-01-01 00:07:00,2,Chest pain,0,0,2025-01-01 00:16:00,PA/NP,Main,0,1,0,2025-01-01 04:50:00,Admitted,ICU,92.7,2025-01-01 07:20:00,3.0,12.0,150.0,436.0,2025-01-01,0,Wednesday,1
1,E000002,P003545,2025-01-01 00:29:00,EMS,58,F,Commercial,2025-01-01 00:41:00,5,Abdominal pain,0,1,2025-01-01 00:45:00,Resident,Main,2,1,0,2025-01-01 01:19:00,Discharged,NaN,92.7,2025-01-01 01:40:00,11.0,16.0,21.0,71.0,2025-01-01,0,Wednesday,1
2,E000003,P121437,2025-01-01 01:51:00,Police,18,M,Commercial,2025-01-01 02:08:00,3,Dizziness / syncope,0,1,2025-01-01 02:09:00,Attending,Main,3,1,1,2025-01-01 05:28:00,Discharged,NaN,92.7,2025-01-01 05:50:00,17.0,18.0,22.0,238.0,2025-01-01,1,Wednesday,1
3,E000004,P003031,2025-01-01 03:04:00,Private vehicle,46,M,Medicaid,2025-01-01 03:11:00,3,Abdominal pain,0,2,2025-01-01 03:29:00,PA/NP,Main,2,1,0,2025-01-01 06:04:00,Discharged,NaN,97.7,2025-01-01 06:28:00,7.0,25.0,24.0,204.0,2025-01-01,3,Wednesday,1
4,E000005,P009228,2025-01-01 04:11:00,Private vehicle,70,M,Medicare,2025-01-01 04:19:00,3,Other,1,3,2025-01-01 04:56:00,Attending,Main,1,1,0,2025-01-01 07:39:00,Observation,Med-Surg,97.7,2025-01-01 09:33:00,8.0,45.0,114.0,321.0,2025-01-01,4,Wednesday,1
5,E000006,P006510,2025-01-01 04:22:00,Private vehicle,41,F,Medicaid,2025-01-01 04:34:00,3,Laceration,0,4,2025-01-01 04:52:00,Attending,Main,0,2,0,2025-01-01 06:53:00,Discharged,NaN,97.7,2025-01-01 07:11:00,12.0,30.0,18.0,169.0,2025-01-01,4,Wednesday,1
6,E000007,P113323,2025-01-01 06:16:00,Private vehicle,64,F,Commercial,2025-01-01 06:29:00,4,Back pain,0,4,2025-01-01 06:46:00,Resident,Main,2,0,0,2025-01-01 07:51:00,Discharged,NaN,97.7,2025-01-01 08:24:00,13.0,29.0,33.0,128.0,2025-01-01,6,Wednesday,1
7,E000008,P003401,2025-01-01 06:45:00,Private vehicle,78,M,Medicare,2025-01-01 06:53:00,2,Injury / trauma,1,4,2025-01-01 07:03:00,Attending,Main,3,1,1,2025-01-01 10:18:00,Admitted,Med-Surg,97.7,2025-01-01 12:51:00,8.0,18.0,153.0,366.0,2025-01-01,6,Wednesday,1
8,E000009,P116671,2025-01-01 08:31:00,Private vehicle,38,M,Commercial,2025-01-01 09:02:00,5,Laceration,0,2,2025-01-01 09:09:00,Resident,Main,0,0,0,2025-01-01 09:37:00,Discharged,NaN,97.7,2025-01-01 10:06:00,31.0,39.0,29.0,95.0,2025-01-01,8,Wednesday,1
9,E000010,P107209,2025-01-01 08:35:00,EMS,64,M,Commercial,2025-01-01 08:48:00,4,Fever / flu-like illness,0,3,2025-01-01 09:01:00,Resident,Main,0,0,0,2025-01-01 10:08:00,Discharged,NaN,97.7,2025-01-01 10:31:00,13.0,26.0,23.0,115.0,2025-01-01,8,Wednesday,1


Things to notice: the timestamps read left to right in time order within each row; `ed_census_at_arrival` is a whole number that describes the department, not the patient; `disposition` is the outcome; and the last eight columns are derived from the timestamps for convenience.

### 3.2 `ed_daily.csv`

One row per day. The first ten rows are 1–10 January.

In [3]:
daily.head(10)

,date,weekday,holiday,ili_index,fast_track_open,visits,ems_arrivals,esi_1_2,admissions,lwbs,lwbs_rate,median_door_to_provider_min,p90_door_to_provider_min,median_ed_los_min,boarding_hours,mean_inpatient_occupancy_pct
0,2025-01-01,Wednesday,1,1.137,0,93,29,14,22,5,0.0538,37.0,76.0,220.0,39.6,90.9
1,2025-01-02,Thursday,0,1.138,0,108,30,25,24,2,0.0185,33.0,94.0,224.0,35.2,85.4
2,2025-01-03,Friday,0,1.139,0,165,39,30,37,5,0.0303,46.0,106.0,248.0,54.8,86.3
3,2025-01-04,Saturday,0,1.140,0,142,34,35,33,4,0.0282,42.0,110.0,254.0,28.5,81.9
4,2025-01-05,Sunday,0,1.141,0,112,21,26,21,3,0.0268,41.0,103.0,218.0,19.8,80.0
5,2025-01-06,Monday,0,1.142,0,106,33,21,27,0,0.0000,34.0,73.0,212.0,30.9,81.2
6,2025-01-07,Tuesday,0,1.143,0,119,32,19,20,0,0.0000,40.0,86.0,222.0,25.4,85.8
7,2025-01-08,Wednesday,0,1.144,0,149,31,39,26,2,0.0134,41.0,108.0,248.0,42.9,85.9
8,2025-01-09,Thursday,0,1.144,0,99,20,16,23,0,0.0000,36.0,87.0,203.0,44.6,89.3
9,2025-01-10,Friday,0,1.145,0,136,42,27,33,2,0.0147,40.0,92.0,222.0,60.7,91.0


### 3.3 `ed_patients.csv`

One row per patient. The first rows are patients shared with the outpatient dataset (low ID numbers); patients new to the hospital have IDs from `P100001` upwards.

In [4]:
patients.head(10)

,patient_id,sex,age,primary_payer,residence_type,distance_miles,known_outpatient
0,P011769,M,70,Medicare,Suburban,5.3,True
1,P000332,M,82,Commercial,Suburban,14.4,True
2,P003140,F,44,Self-pay,Suburban,10.5,True
3,P010700,M,68,Medicare,Urban,1.5,True
4,P004695,M,24,Commercial,Suburban,5.1,True
5,P002726,M,73,Medicare,Urban,3.8,True
6,P007688,F,63,Commercial,Urban,2.0,True
7,P007956,M,74,Medicare,Urban,1.4,True
8,P004534,M,48,Commercial,Suburban,10.6,True
9,P002935,F,45,Medicaid,Urban,5.4,True


## 4. The columns of `ed_visits.csv`, one by one

For each column: what it means, the values it takes, where it comes from in a real ED, and why it matters. The small code cell after each group shows the actual values.

### 4.1 Identifiers and arrival

**`visit_id`** — Unique code per visit (`E000001`, …), assigned in arrival order. Primary key. In a real system this is the ED encounter number.

**`patient_id`** — Pseudonymous patient code. Repeats across rows because patients return: 43% of ED patients visited once in the year, but a small number visited twenty times or more. These "frequent users" are a well-known feature of every real ED. Links to `ed_patients.csv` and, for IDs below `P100000`, to the outpatient `patients.csv`.

**`arrival_datetime`** — Date and time of registration, to the minute. This is the anchor for every time-based analysis.

**`arrival_mode`** — How the patient got to the ED: *Private vehicle* (58%), *EMS* (ambulance, 24%), *Walk-in* (14%), *Police* (4%). Recorded at registration. Ambulance patients are, on average, sicker.

**`age`**, **`sex`**, **`primary_payer`** — As in dataset 1, copied from the patient record. ED patients are younger than the outpatient population (median age 47), and *Self-pay* is more common (11%): the ED is where uninsured patients go.

In [5]:
print("Unique visits:  ", visits["visit_id"].nunique())
print("Unique patients:", visits["patient_id"].nunique())
print("First arrival:  ", visits["arrival_datetime"].min(), "   Last arrival:", visits["arrival_datetime"].max())
print()
print("Arrival mode:")
print(visits["arrival_mode"].value_counts())
print()
print("Visits per patient (how many patients had 1, 2, 3, ... visits):")
print(visits["patient_id"].value_counts().value_counts().sort_index().head(8))

Unique visits:   44934
Unique patients: 18176
First arrival:   2025-01-01 00:04:00    Last arrival: 2025-12-31 23:55:00

Arrival mode:
arrival_mode
Private vehicle    26149
EMS                10816
Walk-in             6329
Police              1640
Name: count, dtype: int64

Visits per patient (how many patients had 1, 2, 3, ... visits):
count
1    7725
2    4263
3    2443
4    1415
5     837
6     571
7     335
8     207
Name: count, dtype: int64


### 4.2 Triage

**`triage_datetime`** — When the triage nurse finished the assessment. Always present: every registered patient is triaged, even those who later leave.

**`esi_level`** — The **Emergency Severity Index**, the five-level triage scale used by most US EDs. Level 1 is resuscitation (about to die without immediate intervention; 2% of visits), level 2 is emergent (high-risk, 17%), level 3 is urgent and needs several resources (42%), level 4 needs one resource (29%), level 5 needs none (9%). The scale is **ordinal**: 1 is more urgent than 2, but the gaps are not equal, so averaging ESI levels is meaningless. Assigned by the triage nurse and stored as a number in the EHR.

**`chief_complaint`** — The presenting problem, grouped at triage into 15 categories (abdominal pain, injury/trauma, chest pain, fever/flu-like illness, shortness of breath, …). In a real EHR this is free text or a pick-list of several hundred items; grouping it is a routine data-preparation step.

**`arrival_vitals_abnormal`** — 1 if any vital sign at arrival (heart rate, blood pressure, oxygen saturation, temperature, respiratory rate) was outside normal limits. A simplification of the full vitals record, which would be a separate table in a real system.

**`ed_census_at_arrival`** — The number of patients physically in the department when this patient arrived. This is not a patient attribute; it describes how **crowded** the ED was at that moment. Real tracking boards compute it continuously. It is one of the most important columns in the file.

In [6]:
print("ESI level:")
print(visits["esi_level"].value_counts().sort_index())
print()
print("Chief complaint:")
print(visits["chief_complaint"].value_counts())
print()
print("ED census at arrival:")
print(visits["ed_census_at_arrival"].describe().round(1))

ESI level:
esi_level
1      849
2     7775
3    18893
4    13184
5     4233
Name: count, dtype: int64

Chief complaint:
chief_complaint
Abdominal pain               5718
Injury / trauma              5271
Other                        4367
Chest pain                   3660
Fever / flu-like illness     3575
Extremity pain / swelling    2955
Shortness of breath          2900
Laceration                   2775
Back pain                    2564
Nausea / vomiting            2306
Headache                     2180
Urinary symptoms             1961
Psychiatric / behavioral     1759
Dizziness / syncope          1723
Altered mental status        1220
Name: count, dtype: int64

ED census at arrival:
count    44934.0
mean        26.0
std         11.9
min          0.0
25%         17.0
50%         25.0
75%         33.0
max         75.0
Name: ed_census_at_arrival, dtype: float64


### 4.3 Treatment

**`provider_datetime`** — First contact with a provider. **Blank for patients who left without being seen** (see `disposition`).

**`provider_type`** — *Attending* physician (47%), *Resident* (28%, a physician in training working under an attending) or *PA/NP* (26%, physician assistant or nurse practitioner). Blank if not seen.

**`treatment_area`** — *Main* (84%), *Fast track* (10%, only from 1 July), *Behavioral* (4%), *Resuscitation* (2%). Blank if not seen.

**`n_labs`**, **`n_imaging`** — Number of laboratory tests and imaging studies ordered during the visit. Counts from the order-entry system. More orders mean a longer visit; they also track acuity.

**`ct_performed`** — 1 if a CT scan was done (about 20% of visits). CT use is a common quality and cost measure in emergency medicine.

In [7]:
print("Provider type:")
print(visits["provider_type"].value_counts(dropna=False))
print()
print("Treatment area:")
print(visits["treatment_area"].value_counts(dropna=False))
print()
print("Orders:")
print(visits[["n_labs", "n_imaging", "ct_performed"]].describe().round(2).loc[["mean", "50%", "max"]])

Provider type:
provider_type
Attending    20623
Resident     12175
PA/NP        11320
NaN            816
Name: count, dtype: int64

Treatment area:
treatment_area
Main             37127
Fast track        4470
Behavioral        1672
Resuscitation      849
NaN                816
Name: count, dtype: int64

Orders:
      n_labs  n_imaging  ct_performed
mean    1.88        0.6          0.19
50%     1.00        0.0          0.00
max    16.00        6.0          1.00


### 4.4 Disposition and departure

**`decision_datetime`** — When the disposition decision was recorded. Blank for patients who left before a decision was made.

**`disposition`** — What happened. *Discharged* home (76%); *Admitted* to an inpatient bed (15%); *Observation* (4%, a short stay under a distinct billing status, usually under 48 hours); *LWBS* (2%, **left without being seen**: registered, usually triaged, then left before provider contact); *Transferred* to another hospital (2%); *AMA* (1%, left against medical advice after being seen); *Eloped* (0.4%, left after being seen but before a decision); *Expired* (0.1%, died in the ED). LWBS is a widely reported quality measure because it means a patient who thought they needed emergency care did not receive it.

**`admit_unit_type`** — For admitted and observation patients, the kind of bed requested: *Med-Surg* (general medical/surgical ward, 53% of admissions), *Telemetry* (continuous cardiac monitoring, 35%), *ICU* (12%). Blank otherwise.

**`inpatient_occupancy_pct`** — Percentage of the hospital's inpatient beds that were occupied at the moment of the disposition decision. This describes the hospital, not the patient, and it is the main reason admitted patients board in the ED: when the wards are full, there is nowhere to send them. Blank when there was no decision.

**`departure_datetime`** — When the patient left the ED. Always present. For visits late on 31 December it can fall on 1 January 2026.

In [8]:
print("Disposition:")
counts = visits["disposition"].value_counts()
print(pd.DataFrame({"visits": counts, "percent": (counts / counts.sum() * 100).round(1)}))
print()
print("Admit unit (admitted and observation patients only):")
print(visits["admit_unit_type"].value_counts())
print()
print("Inpatient occupancy at decision (%):")
print(visits["inpatient_occupancy_pct"].describe().round(1))

Disposition:
             visits  percent
disposition                 
Discharged    33915     75.5
Admitted       6837     15.2
Observation    1906      4.2
LWBS            816      1.8
Transferred     782      1.7
AMA             444      1.0
Eloped          171      0.4
Expired          63      0.1

Admit unit (admitted and observation patients only):
admit_unit_type
Med-Surg     4637
Telemetry    3083
ICU          1023
Name: count, dtype: int64

Inpatient occupancy at decision (%):
count    43947.0
mean        87.1
std          6.5
min         70.0
25%         82.6
50%         87.2
75%         91.9
max         99.0
Name: inpatient_occupancy_pct, dtype: float64


### 4.5 The missing values, and why they are there

This is the first dataset in the programme with blanks. `isna().sum()` counts them per column.

In [9]:
missing = visits.isna().sum()
missing[missing > 0]

provider_datetime              816
provider_type                  816
treatment_area                 816
decision_datetime              987
admit_unit_type              36191
inpatient_occupancy_pct        987
door_to_provider_min           816
decision_to_departure_min      987
dtype: int64

Every blank has a reason:

- **816 visits** have no `provider_datetime`, `provider_type`, `treatment_area` or `door_to_provider_min`. These are the 816 LWBS visits: the patient left before a provider saw them, so there is no provider contact to record.
- **987 visits** have no `decision_datetime`, `inpatient_occupancy_pct` or `decision_to_departure_min`: the 816 LWBS plus the 171 eloped patients, none of whom reached a disposition decision.
- **36,191 visits** have no `admit_unit_type` because the patient was not admitted.

None of this is dirty data. It is **structural missingness**: the field does not apply. The practical consequence is that any analysis of waiting times must decide, and say, what it does with LWBS visits. Excluding them when computing median door-to-provider time, for example, makes the department look faster than the patients experienced it.

In [10]:
check = visits.loc[visits["disposition"] == "LWBS", "provider_datetime"].isna().all()
print("Every LWBS visit has a blank provider timestamp:", check)

Every LWBS visit has a blank provider timestamp: True


### 4.6 Derived columns

The last eight columns are computed from the timestamps and are provided so that the common measures are one `groupby` away. Each can be recomputed, and checking one is a good habit. Timestamps are stored to the minute, so a recomputed interval can differ from the stored one by up to a minute of rounding.

**`door_to_triage_min`** — Minutes from arrival to triage. Median 10.

**`door_to_provider_min`** — Minutes from arrival to provider contact. Median 38 overall, but it varies enormously by acuity and by hour. Blank for LWBS. This is the responsiveness measure that CMS publicly reports for every US hospital.

**`decision_to_departure_min`** — Minutes from disposition decision to departure. For discharged patients it is the paperwork and discharge process (median around 25 minutes). For admitted patients it is **boarding time**, which is the number that hospital leadership tracks most closely. Blank when there was no decision.

**`ed_los_min`** — Total minutes in the ED. Median 227 (just under four hours); much longer for admitted patients.

**`arrival_date`**, **`arrival_hour`**, **`arrival_weekday`**, **`arrival_month`** — Calendar parts of the arrival timestamp, for grouping.

In [11]:
recomputed = (visits["provider_datetime"] - visits["arrival_datetime"]).dt.total_seconds() / 60
seen = visits["provider_datetime"].notna()          # compare only where a provider contact exists
gap = (recomputed[seen] - visits.loc[seen, "door_to_provider_min"]).abs()
print("door_to_provider_min agrees with the timestamps to within one minute:", bool((gap <= 1).all()))
print("largest difference (minutes):", gap.max())
print()
print(visits[["door_to_triage_min", "door_to_provider_min", "decision_to_departure_min", "ed_los_min"]].describe().round(0).loc[["count", "50%", "mean", "max"]])
print()
print("Visits by hour of arrival:")
print(visits["arrival_hour"].value_counts().sort_index().to_string())

door_to_provider_min agrees with the timestamps to within one minute: True
largest difference (minutes): 1.0

       door_to_triage_min  door_to_provider_min  decision_to_departure_min  ed_los_min
count             44934.0               44118.0                    43947.0     44934.0
50%                  10.0                  38.0                       28.0       227.0
mean                 11.0                  46.0                       44.0       255.0
max                  67.0                 674.0                     1059.0      1526.0

Visits by hour of arrival:
arrival_hour
0     1082
1      879
2      705
3      664
4      572
5      620
6      814
7     1256
8     1719
9     2030
10    2323
11    2548
12    2659
13    2832
14    2767
15    2795
16    2802
17    2745
18    2817
19    2690
20    2356
21    2162
22    1724
23    1373


## 5. The columns of `ed_daily.csv`

One row per day, every column an aggregate of `ed_visits.csv` over the visits that **arrived** that day.

| Column | Meaning |
|---|---|
| `date`, `weekday` | Calendar day and its weekday name. |
| `holiday` | 1 on US federal holidays (11 days in 2025). EDs are typically quieter on the holiday and busier the day after. |
| `ili_index` | A seasonal **influenza-like-illness** index, higher in winter. In real analyses this would come from public-health surveillance (the CDC publishes weekly ILI activity). |
| `fast_track_open` | 0 before 1 July 2025, 1 from that day on. |
| `visits`, `ems_arrivals`, `esi_1_2` | Total arrivals, ambulance arrivals, and high-acuity arrivals. |
| `admissions` | Visits ending in *Admitted* or *Observation*. |
| `lwbs`, `lwbs_rate` | Count and share of visits that left without being seen. |
| `median_door_to_provider_min`, `p90_door_to_provider_min` | The day's median and 90th-percentile wait to be seen. |
| `median_ed_los_min` | The day's median length of stay. |
| `boarding_hours` | Total boarding time of admitted patients that day, in hours. A daily total is how boarding is usually reported. |
| `mean_inpatient_occupancy_pct` | Average inpatient occupancy at the decision times of that day's admitted patients. |

Because every column is derived from the visits table, this file can be rebuilt with one `groupby("arrival_date")`, and doing so is a good check of understanding.

In [12]:
daily.describe().round(1).loc[["mean", "min", "50%", "max"]]

,date,holiday,ili_index,fast_track_open,visits,ems_arrivals,esi_1_2,admissions,lwbs,lwbs_rate,median_door_to_provider_min,p90_door_to_provider_min,median_ed_los_min,boarding_hours,mean_inpatient_occupancy_pct
mean,2025-07-01 23:59:59.999999744,0.0,1.0,0.5,123.1,29.6,23.6,24.0,2.2,0.0,37.0,83.4,225.6,41.0,86.7
min,2025-01-01 00:00:00,0.0,0.8,0.0,57.0,10.0,8.0,5.0,0.0,0.0,24.0,49.0,178.0,4.2,71.4
50%,2025-07-02 00:00:00,0.0,1.0,1.0,119.0,29.0,23.0,24.0,2.0,0.0,36.0,81.0,225.0,35.6,86.3
max,2025-12-31 00:00:00,1.0,1.2,1.0,215.0,60.0,49.0,48.0,11.0,0.1,54.0,134.0,277.0,158.8,98.4


## 6. The columns of `ed_patients.csv`

One row per patient: `patient_id`, `sex`, `age`, `primary_payer`, `residence_type` and `distance_miles` with the same meaning as in dataset 1, plus **`known_outpatient`**: True if the patient also appears in the outpatient `patients.csv`. Use this table when you want to count *patients* rather than *visits*, for example to ask how many distinct people account for the ED's frequent-user visits.

In [13]:
print("Patients shared with the outpatient dataset:", patients["known_outpatient"].sum(), f"({patients['known_outpatient'].mean():.0%})")
print()
print(patients["primary_payer"].value_counts(normalize=True).round(3))

Patients shared with the outpatient dataset: 7019 (39%)

primary_payer
Commercial    0.476
Medicaid      0.231
Medicare      0.176
Self-pay      0.117
Name: proportion, dtype: float64


## 7. Summary

The 31 columns of `ed_visits.csv` fall into six groups:

- **Keys and arrival**: `visit_id`, `patient_id`, `arrival_datetime`, `arrival_mode`, and the patient attributes `age`, `sex`, `primary_payer`.
- **Triage**: `triage_datetime`, `esi_level`, `chief_complaint`, `arrival_vitals_abnormal`, and the crowding measure `ed_census_at_arrival`.
- **Treatment**: `provider_datetime`, `provider_type`, `treatment_area`, `n_labs`, `n_imaging`, `ct_performed`.
- **Disposition**: `decision_datetime`, `disposition`, `admit_unit_type`, `inpatient_occupancy_pct`, `departure_datetime`.
- **Derived intervals**: `door_to_triage_min`, `door_to_provider_min`, `decision_to_departure_min`, `ed_los_min`.
- **Calendar parts**: `arrival_date`, `arrival_hour`, `arrival_weekday`, `arrival_month`.

Two habits specific to this dataset: always state what you did with the LWBS visits, and always ask which **unit of analysis** a question needs, the visit, the hour, or the day, before choosing a method. The next sessions use the daily and hourly views to introduce time series, and the visit-level view to ask what drives waiting, leaving and boarding.
